### Filter for non-german names

Many of the items do not have a german name, <br>
unlikely that they are sold in the store under the foreign name.

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import pandas as pd
from transformers import pipeline
from tqdm import tqdm
import math
import torch

### Load the Model

In [3]:
device = 0 if torch.cuda.is_available() else -1
device

0

In [4]:
### set the batch size
batch_size = 64

### load the model for language classification
language_classifier = pipeline(
    "text-classification", 
    model="qanastek/51-languages-classifier",
    batch_size=batch_size,
    device=device  
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/2.63k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/398 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

### Foods

In [ ]:
### load the csv file
path = "/content/drive/MyDrive/Projects/Haltgut/data/german_foods.csv"
df = pd.read_csv(path)

In [9]:
### Add a new column to distinguish german names
df['german_name'] = False

In [12]:
n, _ = df.shape

We iterate over the whole dataset, passing each product´s into the model. <br>
The model returns the language it assigns the highest likelihood the products name to be in. <br>
In case the model returns german we assign the column True, <br>
else we assign it False.

In [20]:
for i in tqdm(range(0, n, batch_size), total=math.ceil(n/batch_size)):

    ### determine last index of the batch
    end = min(i + batch_size, n)

    ### get the batch from the df
    batch_series = df.iloc[i:end]["product_name"]
    batch = batch_series.to_list()
    ### use the classifier to determine if the name is german
    result = language_classifier(
        batch,
        truncation=True
    )
    ### set label as true if german, i.e. "de-DE" and false else
    labels = [r['label'] == 'de-DE' for r in result]
    ### update the dataframe
    df.loc[batch_series.index, "german_name"] = labels

100%|██████████| 4380/4380 [05:55<00:00, 12.33it/s]


In [29]:
df[df["german_name"]==True][:10]

,product_name,categories,countries,german_name
11,Lindt Vollmilch Schokolade,"Dried products, Dried products to be rehydrate...","Frankreich, Germany",True
17,"Fruchtfliegen-Falle, Obstfliegenfalle und Essi...",NaN,en:germany,True
24,BP-ER,"Beverages and beverages preparations, Beverage...","Italien, Germany",True
29,Bio Flohsamenschalen gemahlen,NaN,"Vereinigtes Königreich, Germany",True
31,Schweinebauch mit knoblauch,NaN,"United States, Germany",True
35,Reisprotein,NaN,Germany,True
42,WHEY Gold Standard,NaN,"Irland, Germany",True
46,Bitter Raw Apricot Seeds,Health,"Vereinigte Staaten von Amerika, Germany",True
59,Zucchini Stücke,NaN,en:germany,True
71,Garppa di Chardonnay (nur mehr 1/2 voll),"Getränke und Getränkezubereitungen, Getränke",en:germany,True


In [30]:
df[df["german_name"] == True].shape

(137969, 4)

In [31]:
### save the csv
df.to_csv(path, sep=',')

### Beauty

In [7]:
path = "/content/drive/MyDrive/Projects/Haltgut/data/german_beauty.csv"
df = pd.read_csv(path)

In [10]:
### Add a new column to distinguish german names
df['german_name'] = False

In [8]:
n, _ = df.shape

In [11]:
for i in tqdm(range(0, n, batch_size), total=math.ceil(n/batch_size)):

    ### determine last index of the batch
    end = min(i + batch_size, n)

    ### get the batch from the df
    batch_series = df.iloc[i:end]["product_name"]
    batch = batch_series.to_list()
    ### use the classifier to determine if the name is german
    result = language_classifier(
        batch,
        truncation=True
    )
    ### set label as true if german, i.e. "de-DE" and false else
    labels = [r['label'] == 'de-DE' for r in result]
    ### update the dataframe
    df.loc[batch_series.index, "german_name"] = labels

100%|██████████| 46/46 [00:04<00:00, 11.05it/s]


In [16]:
### save the csv
df.to_csv(path, sep=',')

In [13]:
print(
    f"Datapoints with german name: {len(df[df["german_name"]==True])} \n Datapoints with non-german name: {len(df[df["german_name"]==False])}"
)

Datapoints with german name: 1374 
 Datapoints with non-german name: 1564


In [14]:
df[df["german_name"]==True][:10]

,Unnamed: 0,product_name,categories,countries,german_name
4,482,NEURO STYLE,NaN,en:Germany,True
6,762,All-One Reine Naturseife,NaN,en:Germany,True
7,795,Seife Bronner's,"Non-Food-Produkte, Open Beauty Facts",en:germany,True
8,796,Kosmetik – Teebaum Seife,"Non-Food-Produkte, Open Beauty Facts",en:germany,True
12,1115,schüßler,"Non food products, Open Beauty Facts, Medicine",en:Germany,True
13,1209,Penaten Baby Wundschutzcrème,NaN,en:Germany,True
14,1465,Kamill Hand&Nagelcreme,NaN,en:Germany,True
15,1615,Hygiene Seife,NaN,en:Germany,True
16,1627,Palmolive Hand Wash,NaN,en:Germany,True
17,1647,Peter Schmidinger 5in1 High-Intense micro pow...,Kosmetik,en:Germany,True


In [15]:
df[df["german_name"]==False][:10]

,Unnamed: 0,product_name,categories,countries,german_name
0,7,"Body & Earth, Inc","Non food products, Open Beauty Facts, Cosmetic...","Vereinigtes Königreich, Germany",False
1,444,Nivea Men Fresh Active (0% ACH),NaN,en:Germany,False
2,480,CLEAN CUT®,NaN,en:Germany,False
3,481,Shampoo Blond,en:open-beauty-facts,en:germany,False
5,518,Cocoa butter formula,NaN,en:Germany,False
9,858,Disinfecting wipes,NaN,"Germany,United States",False
10,973,DKNY Stories,Parfüm,en:Germany,False
11,1107,Double Wear Concealer 2C,NaN,Germany,False
20,1707,Nivea men creme,NaN,en:Germany,False
21,1851,Melkfett,NaN,en:Germany,False
